# 🤖 Módulo 06: Introducción a Machine Learning

Tu primer modelo con scikit-learn: preparar datos, entrenar, predecir y evaluar. Regresión y clasificación.

**PandasQuest** · versión web interactiva: [https://rosalesluciano.github.io/pandas-quest/](https://rosalesluciano.github.io/pandas-quest/#/m/m06)

**Cómo funciona:**
1. Ejecuta la celda ⚙️ de abajo (carga las tablas y el corrector).
2. En cada ejercicio escribe tu código y guarda la respuesta en `resultado`.
3. Ejecuta la celda: `comprobar()` te dirá si está bien. 🎯

¿Atascado? `ver_solucion("id")` muestra la solución (¡pero inténtalo antes! 😉)

In [ ]:
#@title ⚙️ 1) Ejecuta esta celda primero (carga datos y corrector) { display-mode: "form" }
"""PandasQuest - motor de corrección.

Se usa igual en el navegador (Pyodide), en los notebooks de Colab y en los tests.
Compara el `resultado` del alumno contra el de la solución oficial con tolerancia
a diferencias irrelevantes (tipos int/float, nombres de índice, dtype string, etc.).
"""
import math

import numpy as np
import pandas as pd


def _norm_s(s):
    dt = s.dtype
    if isinstance(dt, pd.CategoricalDtype) or pd.api.types.is_string_dtype(dt):
        s = s.astype(object)
    elif pd.api.types.is_extension_array_dtype(dt):
        if pd.api.types.is_numeric_dtype(dt) and not pd.api.types.is_bool_dtype(dt):
            s = s.astype("float64")
        else:
            s = s.astype(object)
    if s.dtype == object:
        s = s.where(s.notna(), None)
    return s


def _norm_index(idx):
    if isinstance(idx, pd.MultiIndex):
        return idx
    if isinstance(idx, pd.CategoricalIndex) or pd.api.types.is_string_dtype(idx.dtype):
        return pd.Index(list(idx), dtype=object)
    return idx


def _norm(o):
    if isinstance(o, pd.Series):
        o = _norm_s(o.copy())
    else:
        cols = list(o.columns)
        o = pd.DataFrame({i: _norm_s(o.iloc[:, i]) for i in range(o.shape[1])}, index=o.index)
        o.columns = _norm_index(pd.Index(cols))
    o.index = _norm_index(o.index)
    return o


def _sort(o, ignore_index):
    if ignore_index:
        o = o.reset_index(drop=True)
        try:
            if isinstance(o, pd.Series):
                o = o.sort_values(kind="mergesort")
            else:
                o = o.sort_values(by=list(o.columns), kind="mergesort")
        except TypeError:
            key = o.astype(str) if isinstance(o, pd.Series) else o.astype(str).agg("|".join, axis=1)
            o = o.iloc[np.argsort(key.to_numpy(), kind="mergesort")]
        return o.reset_index(drop=True)
    try:
        return o.sort_index()
    except TypeError:
        return o


def _tipo(x):
    nombres = {pd.DataFrame: "un DataFrame", pd.Series: "una Series"}
    return nombres.get(type(x), f"un {type(x).__name__}")


def _scalar_eq(u, e):
    if isinstance(e, (np.generic,)):
        e = e.item()
    if isinstance(u, (np.generic,)):
        u = u.item()
    if isinstance(e, float) or isinstance(u, float):
        try:
            u, e = float(u), float(e)
        except (TypeError, ValueError):
            return False
        if math.isnan(e):
            return math.isnan(u)
        return math.isclose(u, e, rel_tol=1e-6, abs_tol=1e-6)
    if isinstance(e, (list, tuple)) and isinstance(u, (list, tuple, np.ndarray, pd.Index)):
        u = list(u)
        return len(u) == len(e) and all(_scalar_eq(a, b) for a, b in zip(u, e))
    try:
        return bool(u == e)
    except Exception:
        return False


def _array_compare(u, e):
    if isinstance(u, (pd.DataFrame, pd.Series)):
        u = u.to_numpy()
    try:
        ua = np.asarray(u)
    except Exception:
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if not isinstance(u, np.ndarray) and not isinstance(u, (list, tuple)):
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if ua.shape != e.shape:
        return False, f"Forma distinta: tu array tiene shape {ua.shape} y se esperaba {e.shape}."
    num = np.issubdtype(e.dtype, np.number) or e.dtype == bool
    try:
        if num and (np.issubdtype(ua.dtype, np.number) or ua.dtype == bool):
            ok = np.allclose(ua.astype(float), e.astype(float), rtol=1e-6, atol=1e-6, equal_nan=True)
        else:
            ok = ua.tolist() == e.tolist()
    except Exception:
        ok = False
    return (True, "") if ok else (False, "La forma es correcta pero los valores no coinciden. Revisa el cálculo.")


def pq_compare(u, e, ordered=True, ignore_index=False):
    """Devuelve (ok, mensaje)."""
    if isinstance(e, np.ndarray):
        return _array_compare(u, e)
    if isinstance(e, (pd.DataFrame, pd.Series)):
        if not isinstance(u, type(e)):
            extra = ""
            if isinstance(e, pd.Series) and isinstance(u, pd.DataFrame):
                extra = " Pista: con una sola columna usa df['col'] (corchetes simples)."
            if isinstance(e, pd.DataFrame) and isinstance(u, pd.Series):
                extra = " Pista: usa doble corchete df[['col']] o .reset_index() / .to_frame()."
            return False, f"Tu resultado es {_tipo(u)}, pero se esperaba {_tipo(e)}.{extra}"
        if isinstance(e, pd.DataFrame):
            uc, ec = [str(c) for c in u.columns], [str(c) for c in e.columns]
            if uc != ec:
                if sorted(uc) == sorted(ec):
                    return False, f"Tienes las columnas correctas pero en otro orden. Se esperaba: {ec}"
                faltan = [c for c in ec if c not in uc]
                sobran = [c for c in uc if c not in ec]
                msg = "Las columnas no coinciden."
                if faltan:
                    msg += f" Faltan: {faltan}."
                if sobran:
                    msg += f" Sobran: {sobran}."
                return False, msg
        if u.shape != e.shape:
            return False, f"Forma distinta: tu resultado tiene {u.shape} y se esperaba {e.shape} (filas, columnas)."
        u2, e2 = _norm(u), _norm(e)
        if not ordered:
            u2, e2 = _sort(u2, ignore_index), _sort(e2, ignore_index)
        elif ignore_index:
            u2, e2 = u2.reset_index(drop=True), e2.reset_index(drop=True)
        kw = dict(check_dtype=False, check_names=False, check_index_type=False,
                  check_exact=False, rtol=1e-6, atol=1e-6)
        try:
            if isinstance(e2, pd.DataFrame):
                pd.testing.assert_frame_equal(u2, e2, check_column_type=False, check_freq=False, **kw)
            else:
                pd.testing.assert_series_equal(u2, e2, check_freq=False, **kw)
        except AssertionError as err:
            txt = str(err)
            if "index" in txt.lower() and "values are different" in txt.lower() and "iloc" not in txt:
                return False, "Los valores parecen correctos pero el índice no coincide. ¿Hace falta ordenar o usar reset_index()?"
            if ordered and not ignore_index:
                try:
                    pd.testing.assert_frame_equal(
                        u2.to_frame() if isinstance(u2, pd.Series) else u2,
                        e2.to_frame() if isinstance(e2, pd.Series) else e2,
                        check_like=True, **kw)
                except AssertionError:
                    pass
                else:
                    return False, "Mismos datos, pero en otro orden. Revisa cómo ordenas las filas."
            return False, "Los valores no coinciden con lo esperado. Revisa la lógica y vuelve a intentarlo."
        return True, ""
    if isinstance(u, (pd.DataFrame, pd.Series)):
        return False, f"Tu resultado es {_tipo(u)}, pero se esperaba un valor simple ({type(e).__name__}). ¿Te falta .iloc[0], .sum() o similar?"
    if _scalar_eq(u, e):
        return True, ""
    return False, f"Tu resultado es {u!r}, pero no es el valor esperado."


# ---------------------------------------------------------------- gráficos
def _plt():
    import matplotlib.pyplot as plt
    return plt


def _fig():
    plt = _plt()
    return plt.gcf() if plt.get_fignums() else None


def _ax(i=0):
    f = _fig()
    if f is None or len(f.axes) <= i:
        raise LookupError("no hay ningún gráfico dibujado")
    return f.axes[i]


def _legend(ax=None):
    ax = ax or _ax()
    lg = ax.get_legend()
    return [t.get_text() for t in lg.get_texts()] if lg else []


def _heights(ax=None):
    return [round(float(p.get_height()), 6) for p in (ax or _ax()).patches]


def _widths(ax=None):
    return [round(float(p.get_width()), 6) for p in (ax or _ax()).patches]


def _xticks(ax=None):
    return [t.get_text() for t in (ax or _ax()).get_xticklabels()]


def _hex(color):
    import matplotlib.colors as mcolors
    return mcolors.to_hex(color)


_HELPERS = {"_fig": _fig, "_ax": _ax, "_legend": _legend, "_heights": _heights,
            "_widths": _widths, "_xticks": _xticks, "_hex": _hex}


def pq_prepare():
    """Deja matplotlib limpio antes de ejecutar código (si está cargado)."""
    import sys
    if "matplotlib.pyplot" in sys.modules:
        plt = sys.modules["matplotlib.pyplot"]
        plt.close("all")
        plt.show = lambda *a, **k: None


def pq_check(ns, get_expected, opts):
    """Evalúa el ejercicio.

    ns = namespace del alumno. get_expected = función que devuelve el namespace de la
    solución (solo se ejecuta si hace falta, para no pisar los gráficos del alumno).
    """
    custom = opts.get("custom")
    if custom:
        if opts.get("plot") and _fig() is None:
            return False, "Todavía no hay ningún gráfico. ¡Dibuja algo! 🎨"
        env = dict(_HELPERS)
        env.update(ns)
        try:
            ok = bool(eval(custom, env))
        except Exception as err:
            return False, f"{opts.get('custom_msg', 'Todavía no es correcto.')} ({type(err).__name__}: {err})"
        return ok, "" if ok else opts.get("custom_msg", "Todavía no es correcto. ¡Sigue intentándolo!")
    if "resultado" not in ns:
        return False, "Guarda tu respuesta en la variable `resultado`."
    exp_ns = get_expected() if callable(get_expected) else get_expected
    return pq_compare(ns["resultado"], exp_ns["resultado"],
                      ordered=opts.get("ordered", True),
                      ignore_index=opts.get("ignore_index", False))


_ERR_TIPS = [
    ("truth value of a Series is ambiguous", "Para combinar condiciones usa & (y), | (o) y pon cada condición entre paréntesis."),
    ("KeyError", "Revisa que el nombre de la columna esté bien escrito (mayúsculas, tildes, espacios)."),
    ("NameError", "Estás usando una variable que no existe. ¿Un error de tipeo?"),
    ("SyntaxError", "Hay un error de sintaxis: revisa paréntesis, comillas y que la línea `resultado = ...` esté completa."),
    ("AttributeError", "Ese método o atributo no existe para este objeto. Revisa el nombre."),
    ("TypeError", "Algún tipo de dato no encaja (¿texto mezclado con números?)."),
    ("IndentationError", "Revisa la sangría (espacios al inicio de la línea)."),
    ("MergeError", "Revisa las columnas que usas para unir (on / left_on / right_on)."),
]


def pq_error_tip(msg):
    for key, tip in _ERR_TIPS:
        if key in msg:
            return tip
    return ""


_SETUP = 'import pandas as pd\nimport numpy as np\n\nclientes = pd.DataFrame({\n    "cliente_id": range(1, 13),\n    "nombre": ["Ana", "Bruno", "Carla", "Diego", "Elena", "Facundo",\n               "Gabriela", "Hugo", "Inés", "Julián", "Karina", "Lucas"],\n    "ciudad": ["Madrid", "Buenos Aires", "CDMX", "Bogotá", "Madrid", "Lima",\n               "Buenos Aires", "CDMX", "Santiago", "Bogotá", "Madrid", "Lima"],\n    "pais": ["España", "Argentina", "México", "Colombia", "España", "Perú",\n             "Argentina", "México", "Chile", "Colombia", "España", "Perú"],\n    "edad": [34, 28, 45, 23, 51, 39, 30, 27, 42, 36, 25, 48],\n    "segmento": ["Premium", "Básico", "Premium", "Básico", "Premium", "Estándar",\n                 "Estándar", "Básico", "Premium", "Estándar", "Básico", "Estándar"],\n    "email": ["ana@mail.com", None, "carla@mail.com", "diego@mail.com", None, "facu@mail.com",\n              "gabi@mail.com", None, "ines@mail.com", "julian@mail.com", "kari@mail.com", "lucas@mail.com"],\n    "fecha_registro": pd.to_datetime([\n        "2023-01-15", "2023-02-03", "2023-02-20", "2023-03-11", "2023-04-02", "2023-05-19",\n        "2023-06-07", "2023-07-23", "2023-08-30", "2023-09-14", "2023-10-05", "2023-11-28"]),\n})\n\nproductos = pd.DataFrame({\n    "producto_id": range(1, 11),\n    "producto": ["Laptop", "Mouse", "Teclado", "Monitor", "Auriculares",\n                 "Webcam", "Silla gamer", "Escritorio", "Tablet", "Micrófono"],\n    "categoria": ["Computación", "Accesorios", "Accesorios", "Computación", "Audio",\n                  "Accesorios", "Muebles", "Muebles", "Computación", "Audio"],\n    "precio": [1200.0, 25.5, 45.0, 300.0, 80.0, 60.0, 250.0, 180.0, 450.0, 95.0],\n    "stock": [5, 150, 80, 20, 60, 0, 12, 7, 15, 0],\n})\n\n_estados = ["entregado", "enviado", "entregado", "cancelado", "entregado", "pendiente"]\npedidos = pd.DataFrame({\n    "pedido_id": range(1001, 1031),\n    "cliente_id": [(i * 7) % 11 + 1 for i in range(29)] + [99],\n    "producto_id": [(i * 3) % 10 + 1 for i in range(30)],\n    "cantidad": [(i % 4) + 1 for i in range(30)],\n    "fecha": pd.date_range("2024-01-03", periods=30, freq="5D"),\n    "estado": [_estados[i % 6] for i in range(30)],\n})\n\nempleados = pd.DataFrame({\n    "emp_id": range(1, 11),\n    "nombre": ["Marta", "Pablo", "Sofía", "Tomás", "Valeria",\n               "Andrés", "Lucía", "Martín", "Paula", "Ramiro"],\n    "depto": ["Dirección", "Ventas", "Ventas", "Ventas", "IT",\n              "IT", "IT", "Marketing", "Marketing", "Ventas"],\n    "salario": [9000, 4200, 3900, 4000, 5200, 4800, 6100, 3700, 4100, 3500],\n    "jefe_id": pd.array([None, 1, 2, 2, 1, 5, 5, 1, 8, 2], dtype="Int64"),\n    "fecha_ingreso": pd.to_datetime([\n        "2015-03-01", "2018-06-15", "2019-01-10", "2020-09-01", "2017-11-20",\n        "2021-02-01", "2016-08-08", "2022-04-18", "2019-07-07", "2023-01-09"]),\n})\n\n_i = np.arange(90)\n_f = pd.date_range("2024-01-01", periods=90, freq="D")\n_v = 1000 + 8 * _i + 250 * (_f.dayofweek >= 5) + ((_i * 37) % 11) * 20\n_v = np.where(np.isin(_i, [20, 55, 77]), _v + 1500, _v)\nventas_diarias = pd.DataFrame({\n    "fecha": _f,\n    "ventas": _v.astype(float),\n    "visitas": (300 + (_i * 13) % 97 + 2 * _i).astype(int),\n})\n\nsucio = pd.DataFrame({\n    "id": [1, 2, 2, 3, 4, 5, 6, 7],\n    "nombre": ["  ana ", "BRUNO", "BRUNO", "carla", None, "Diego  ", "elena", "facundo"],\n    "edad": ["34", "28", "28", "n/d", "23", "51", None, "39"],\n    "ciudad": ["madrid", "Buenos Aires", "Buenos Aires", " santiago", "bogotá", "Madrid ", "madrid", "LIMA"],\n    "monto": ["$1,200.50", "$300", "$300", "$45.00", None, "$80.5", "$99", "$1,000"],\n})\ndel _i, _f, _v, _estados\n\n_rng = np.random.default_rng(42)\n_n = 200\n_m2 = _rng.integers(35, 220, _n)\n_hab = np.clip(_m2 // 35 + _rng.integers(-1, 2, _n), 1, 6)\n_ant = _rng.integers(0, 60, _n)\n_barrio = _rng.choice(["Centro", "Norte", "Sur"], _n)\n_extra = np.select([_barrio == "Centro", _barrio == "Norte"], [40000, 15000], 0)\ncasas = pd.DataFrame({\n    "m2": _m2, "habitaciones": _hab, "antiguedad": _ant, "barrio": _barrio,\n    "precio": np.round(30000 + _m2 * 1800 + _hab * 5000 - _ant * 900 + _extra\n                       + _rng.normal(0, 15000, _n), -2),\n})\n_n = 300\nchurn = pd.DataFrame({\n    "meses_cliente": _rng.integers(1, 72, _n),\n    "cargo_mensual": np.round(_rng.uniform(15, 120, _n), 2),\n    "reclamos": _rng.poisson(1.2, _n),\n    "contrato": _rng.choice(["Mensual", "Anual"], _n, p=[0.6, 0.4]),\n})\n_logit = (-1.0 - 0.05 * churn["meses_cliente"] + 0.03 * churn["cargo_mensual"]\n          + 0.6 * churn["reclamos"] + np.where(churn["contrato"] == "Mensual", 1.2, -0.8))\nchurn["abandona"] = (_rng.uniform(0, 1, _n) < 1 / (1 + np.exp(-_logit))).astype(int)\ndel _rng, _n, _m2, _hab, _ant, _barrio, _extra, _logit\n'
_SOL_B64 = 'eyJtbDAxIjogeyJzb2x1dGlvbiI6ICJYID0gY2FzYXNbWydtMicsICdoYWJpdGFjaW9uZXMnLCAnYW50aWd1ZWRhZCddXVxueSA9IGNhc2FzWydwcmVjaW8nXSIsICJjaGVjayI6IHsiY3VzdG9tIjogImxpc3QoWC5jb2x1bW5zKSA9PSBbJ20yJywgJ2hhYml0YWNpb25lcycsICdhbnRpZ3VlZGFkJ10gYW5kIGxlbihYKSA9PSAyMDAgYW5kIHkubmFtZSA9PSAncHJlY2lvJyBhbmQgbGVuKHkpID09IDIwMCIsICJjdXN0b21fbXNnIjogIlggZGViZSB0ZW5lciBsYXMgY29sdW1uYXMgbTIsIGhhYml0YWNpb25lcyB5IGFudGlndWVkYWQgKGVuIGVzZSBvcmRlbikgZSB5IGRlYmUgc2VyIGxhIGNvbHVtbmEgcHJlY2lvLiJ9LCAieHAiOiAxMH0sICJtbDAyIjogeyJzb2x1dGlvbiI6ICJmcm9tIHNrbGVhcm4ubW9kZWxfc2VsZWN0aW9uIGltcG9ydCB0cmFpbl90ZXN0X3NwbGl0XG5cblggPSBjYXNhc1tbJ20yJywgJ2hhYml0YWNpb25lcycsICdhbnRpZ3VlZGFkJ11dXG55ID0gY2FzYXNbJ3ByZWNpbyddXG5YX3RyYWluLCBYX3Rlc3QsIHlfdHJhaW4sIHlfdGVzdCA9IHRyYWluX3Rlc3Rfc3BsaXQoWCwgeSwgdGVzdF9zaXplPTAuMiwgcmFuZG9tX3N0YXRlPTQyKVxucmVzdWx0YWRvID0gWF90ZXN0IiwgImNoZWNrIjoge30sICJ4cCI6IDIwfSwgIm1sMDMiOiB7InNvbHV0aW9uIjogInJlc3VsdGFkbyA9IHBkLmdldF9kdW1taWVzKGNhc2FzLCBjb2x1bW5zPVsnYmFycmlvJ10sIGRyb3BfZmlyc3Q9VHJ1ZSwgZHR5cGU9aW50KSIsICJjaGVjayI6IHt9LCAieHAiOiAzNX0sICJtbDA0IjogeyJzb2x1dGlvbiI6ICJmcm9tIHNrbGVhcm4ubW9kZWxfc2VsZWN0aW9uIGltcG9ydCB0cmFpbl90ZXN0X3NwbGl0XG5cblggPSBjYXNhc1tbJ20yJywgJ2hhYml0YWNpb25lcycsICdhbnRpZ3VlZGFkJ11dXG55ID0gY2FzYXNbJ3ByZWNpbyddXG5YX3RyYWluLCBYX3Rlc3QsIHlfdHJhaW4sIHlfdGVzdCA9IHRyYWluX3Rlc3Rfc3BsaXQoWCwgeSwgdGVzdF9zaXplPTAuMiwgcmFuZG9tX3N0YXRlPTQyKVxuXG5mcm9tIHNrbGVhcm4ubGluZWFyX21vZGVsIGltcG9ydCBMaW5lYXJSZWdyZXNzaW9uXG5cbm1vZGVsbyA9IExpbmVhclJlZ3Jlc3Npb24oKVxubW9kZWxvLmZpdChYX3RyYWluLCB5X3RyYWluKVxuXG5yZXN1bHRhZG8gPSBtb2RlbG8uY29lZl8iLCAiY2hlY2siOiB7fSwgInhwIjogMjB9LCAibWwwNSI6IHsic29sdXRpb24iOiAiZnJvbSBza2xlYXJuLm1vZGVsX3NlbGVjdGlvbiBpbXBvcnQgdHJhaW5fdGVzdF9zcGxpdFxuXG5YID0gY2FzYXNbWydtMicsICdoYWJpdGFjaW9uZXMnLCAnYW50aWd1ZWRhZCddXVxueSA9IGNhc2FzWydwcmVjaW8nXVxuWF90cmFpbiwgWF90ZXN0LCB5X3RyYWluLCB5X3Rlc3QgPSB0cmFpbl90ZXN0X3NwbGl0KFgsIHksIHRlc3Rfc2l6ZT0wLjIsIHJhbmRvbV9zdGF0ZT00MilcblxuZnJvbSBza2xlYXJuLmxpbmVhcl9tb2RlbCBpbXBvcnQgTGluZWFyUmVncmVzc2lvblxubW9kZWxvID0gTGluZWFyUmVncmVzc2lvbigpXG5tb2RlbG8uZml0KFhfdHJhaW4sIHlfdHJhaW4pXG5cbnJlc3VsdGFkbyA9IG1vZGVsby5wcmVkaWN0KFhfdGVzdCkiLCAiY2hlY2siOiB7fSwgInhwIjogMjB9LCAibWwwNiI6IHsic29sdXRpb24iOiAiZnJvbSBza2xlYXJuLm1vZGVsX3NlbGVjdGlvbiBpbXBvcnQgdHJhaW5fdGVzdF9zcGxpdFxuXG5YID0gY2FzYXNbWydtMicsICdoYWJpdGFjaW9uZXMnLCAnYW50aWd1ZWRhZCddXVxueSA9IGNhc2FzWydwcmVjaW8nXVxuWF90cmFpbiwgWF90ZXN0LCB5X3RyYWluLCB5X3Rlc3QgPSB0cmFpbl90ZXN0X3NwbGl0KFgsIHksIHRlc3Rfc2l6ZT0wLjIsIHJhbmRvbV9zdGF0ZT00MilcblxuZnJvbSBza2xlYXJuLmxpbmVhcl9tb2RlbCBpbXBvcnQgTGluZWFyUmVncmVzc2lvblxubW9kZWxvID0gTGluZWFyUmVncmVzc2lvbigpXG5tb2RlbG8uZml0KFhfdHJhaW4sIHlfdHJhaW4pXG5wcmVkaWNjaW9uZXMgPSBtb2RlbG8ucHJlZGljdChYX3Rlc3QpXG5cbmZyb20gc2tsZWFybi5tZXRyaWNzIGltcG9ydCBtZWFuX2Fic29sdXRlX2Vycm9yXG5yZXN1bHRhZG8gPSBtZWFuX2Fic29sdXRlX2Vycm9yKHlfdGVzdCwgcHJlZGljY2lvbmVzKSIsICJjaGVjayI6IHt9LCAieHAiOiAyMH0sICJtbDA3IjogeyJzb2x1dGlvbiI6ICJmcm9tIHNrbGVhcm4ubW9kZWxfc2VsZWN0aW9uIGltcG9ydCB0cmFpbl90ZXN0X3NwbGl0XG5cblggPSBjYXNhc1tbJ20yJywgJ2hhYml0YWNpb25lcycsICdhbnRpZ3VlZGFkJ11dXG55ID0gY2FzYXNbJ3ByZWNpbyddXG5YX3RyYWluLCBYX3Rlc3QsIHlfdHJhaW4sIHlfdGVzdCA9IHRyYWluX3Rlc3Rfc3BsaXQoWCwgeSwgdGVzdF9zaXplPTAuMiwgcmFuZG9tX3N0YXRlPTQyKVxuXG5mcm9tIHNrbGVhcm4ubGluZWFyX21vZGVsIGltcG9ydCBMaW5lYXJSZWdyZXNzaW9uXG5tb2RlbG8gPSBMaW5lYXJSZWdyZXNzaW9uKClcbm1vZGVsby5maXQoWF90cmFpbiwgeV90cmFpbilcbnByZWRpY2Npb25lcyA9IG1vZGVsby5wcmVkaWN0KFhfdGVzdClcblxuZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0IHIyX3Njb3JlXG5yZXN1bHRhZG8gPSByMl9zY29yZSh5X3Rlc3QsIHByZWRpY2Npb25lcykiLCAiY2hlY2siOiB7fSwgInhwIjogMzV9LCAibWwwOCI6IHsic29sdXRpb24iOiAiZnJvbSBza2xlYXJuLm1vZGVsX3NlbGVjdGlvbiBpbXBvcnQgdHJhaW5fdGVzdF9zcGxpdFxuXG5YID0gY2FzYXNbWydtMicsICdoYWJpdGFjaW9uZXMnLCAnYW50aWd1ZWRhZCddXVxueSA9IGNhc2FzWydwcmVjaW8nXVxuWF90cmFpbiwgWF90ZXN0LCB5X3RyYWluLCB5X3Rlc3QgPSB0cmFpbl90ZXN0X3NwbGl0KFgsIHksIHRlc3Rfc2l6ZT0wLjIsIHJhbmRvbV9zdGF0ZT00MilcblxuZnJvbSBza2xlYXJuLmxpbmVhcl9tb2RlbCBpbXBvcnQgTGluZWFyUmVncmVzc2lvblxubW9kZWxvID0gTGluZWFyUmVncmVzc2lvbigpXG5tb2RlbG8uZml0KFhfdHJhaW4sIHlfdHJhaW4pXG5wcmVkaWNjaW9uZXMgPSBtb2RlbG8ucHJlZGljdChYX3Rlc3QpXG5cbm51ZXZhID0gcGQuRGF0YUZyYW1lKFt7J20yJzogOTAsICdoYWJpdGFjaW9uZXMnOiAzLCAnYW50aWd1ZWRhZCc6IDEwfV0pXG5yZXN1bHRhZG8gPSBtb2RlbG8ucHJlZGljdChudWV2YSlbMF0iLCAiY2hlY2siOiB7fSwgInhwIjogMjB9LCAibWwwOSI6IHsic29sdXRpb24iOiAiZnJvbSBza2xlYXJuLm1vZGVsX3NlbGVjdGlvbiBpbXBvcnQgdHJhaW5fdGVzdF9zcGxpdFxuXG5YID0gY2h1cm5bWydtZXNlc19jbGllbnRlJywgJ2NhcmdvX21lbnN1YWwnLCAncmVjbGFtb3MnXV1cbnkgPSBjaHVyblsnYWJhbmRvbmEnXVxuWF90cmFpbiwgWF90ZXN0LCB5X3RyYWluLCB5X3Rlc3QgPSB0cmFpbl90ZXN0X3NwbGl0KFgsIHksIHRlc3Rfc2l6ZT0wLjI1LCByYW5kb21fc3RhdGU9NDIsIHN0cmF0aWZ5PXkpXG5cbmZyb20gc2tsZWFybi5saW5lYXJfbW9kZWwgaW1wb3J0IExvZ2lzdGljUmVncmVzc2lvblxuZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0IGFjY3VyYWN5X3Njb3JlXG5tb2RlbG8gPSBMb2dpc3RpY1JlZ3Jlc3Npb24obWF4X2l0ZXI9MTAwMClcbm1vZGVsby5maXQoWF90cmFpbiwgeV90cmFpbilcbnByZWQgPSBtb2RlbG8ucHJlZGljdChYX3Rlc3QpXG5yZXN1bHRhZG8gPSBhY2N1cmFjeV9zY29yZSh5X3Rlc3QsIHByZWQpIiwgImNoZWNrIjoge30sICJ4cCI6IDM1fSwgIm1sMTAiOiB7InNvbHV0aW9uIjogImZyb20gc2tsZWFybi5tb2RlbF9zZWxlY3Rpb24gaW1wb3J0IHRyYWluX3Rlc3Rfc3BsaXRcblxuWCA9IGNodXJuW1snbWVzZXNfY2xpZW50ZScsICdjYXJnb19tZW5zdWFsJywgJ3JlY2xhbW9zJ11dXG55ID0gY2h1cm5bJ2FiYW5kb25hJ11cblhfdHJhaW4sIFhfdGVzdCwgeV90cmFpbiwgeV90ZXN0ID0gdHJhaW5fdGVzdF9zcGxpdChYLCB5LCB0ZXN0X3NpemU9MC4yNSwgcmFuZG9tX3N0YXRlPTQyLCBzdHJhdGlmeT15KVxuXG5mcm9tIHNrbGVhcm4ubGluZWFyX21vZGVsIGltcG9ydCBMb2dpc3RpY1JlZ3Jlc3Npb25cbmZyb20gc2tsZWFybi5tZXRyaWNzIGltcG9ydCBhY2N1cmFjeV9zY29yZVxubW9kZWxvID0gTG9naXN0aWNSZWdyZXNzaW9uKG1heF9pdGVyPTEwMDApXG5tb2RlbG8uZml0KFhfdHJhaW4sIHlfdHJhaW4pXG5wcmVkID0gbW9kZWxvLnByZWRpY3QoWF90ZXN0KVxuXG5mcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgY29uZnVzaW9uX21hdHJpeFxucmVzdWx0YWRvID0gY29uZnVzaW9uX21hdHJpeCh5X3Rlc3QsIHByZWQpIiwgImNoZWNrIjoge30sICJ4cCI6IDM1fSwgIm1sMTEiOiB7InNvbHV0aW9uIjogImZyb20gc2tsZWFybi5tb2RlbF9zZWxlY3Rpb24gaW1wb3J0IHRyYWluX3Rlc3Rfc3BsaXRcblxuWCA9IGNodXJuW1snbWVzZXNfY2xpZW50ZScsICdjYXJnb19tZW5zdWFsJywgJ3JlY2xhbW9zJ11dXG55ID0gY2h1cm5bJ2FiYW5kb25hJ11cblhfdHJhaW4sIFhfdGVzdCwgeV90cmFpbiwgeV90ZXN0ID0gdHJhaW5fdGVzdF9zcGxpdChYLCB5LCB0ZXN0X3NpemU9MC4yNSwgcmFuZG9tX3N0YXRlPTQyLCBzdHJhdGlmeT15KVxuXG5mcm9tIHNrbGVhcm4udHJlZSBpbXBvcnQgRGVjaXNpb25UcmVlQ2xhc3NpZmllclxuZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0IGFjY3VyYWN5X3Njb3JlXG5cbmFyYm9sID0gRGVjaXNpb25UcmVlQ2xhc3NpZmllcihtYXhfZGVwdGg9MywgcmFuZG9tX3N0YXRlPTApXG5hcmJvbC5maXQoWF90cmFpbiwgeV90cmFpbilcbnJlc3VsdGFkbyA9IGFjY3VyYWN5X3Njb3JlKHlfdGVzdCwgYXJib2wucHJlZGljdChYX3Rlc3QpKSIsICJjaGVjayI6IHt9LCAieHAiOiAzNX0sICJtbDEyIjogeyJzb2x1dGlvbiI6ICJmcm9tIHNrbGVhcm4ubW9kZWxfc2VsZWN0aW9uIGltcG9ydCB0cmFpbl90ZXN0X3NwbGl0XG5cblggPSBjaHVybltbJ21lc2VzX2NsaWVudGUnLCAnY2FyZ29fbWVuc3VhbCcsICdyZWNsYW1vcyddXVxueSA9IGNodXJuWydhYmFuZG9uYSddXG5YX3RyYWluLCBYX3Rlc3QsIHlfdHJhaW4sIHlfdGVzdCA9IHRyYWluX3Rlc3Rfc3BsaXQoWCwgeSwgdGVzdF9zaXplPTAuMjUsIHJhbmRvbV9zdGF0ZT00Miwgc3RyYXRpZnk9eSlcblxuZnJvbSBza2xlYXJuLnBpcGVsaW5lIGltcG9ydCBtYWtlX3BpcGVsaW5lXG5mcm9tIHNrbGVhcm4ucHJlcHJvY2Vzc2luZyBpbXBvcnQgU3RhbmRhcmRTY2FsZXJcbmZyb20gc2tsZWFybi5uZWlnaGJvcnMgaW1wb3J0IEtOZWlnaGJvcnNDbGFzc2lmaWVyXG5mcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgYWNjdXJhY3lfc2NvcmVcblxucGlwZSA9IG1ha2VfcGlwZWxpbmUoU3RhbmRhcmRTY2FsZXIoKSwgS05laWdoYm9yc0NsYXNzaWZpZXIobl9uZWlnaGJvcnM9NSkpXG5waXBlLmZpdChYX3RyYWluLCB5X3RyYWluKVxucmVzdWx0YWRvID0gYWNjdXJhY3lfc2NvcmUoeV90ZXN0LCBwaXBlLnByZWRpY3QoWF90ZXN0KSkiLCAiY2hlY2siOiB7fSwgInhwIjogNTB9LCAibWwxMyI6IHsic29sdXRpb24iOiAiZnJvbSBza2xlYXJuLm1vZGVsX3NlbGVjdGlvbiBpbXBvcnQgY3Jvc3NfdmFsX3Njb3JlXG5mcm9tIHNrbGVhcm4udHJlZSBpbXBvcnQgRGVjaXNpb25UcmVlQ2xhc3NpZmllclxuXG5YID0gY2h1cm5bWydtZXNlc19jbGllbnRlJywgJ2NhcmdvX21lbnN1YWwnLCAncmVjbGFtb3MnXV1cbnkgPSBjaHVyblsnYWJhbmRvbmEnXVxuXG5zY29yZXMgPSBjcm9zc192YWxfc2NvcmUoRGVjaXNpb25UcmVlQ2xhc3NpZmllcihtYXhfZGVwdGg9MywgcmFuZG9tX3N0YXRlPTApLCBYLCB5LCBjdj01KVxucmVzdWx0YWRvID0gc2NvcmVzLm1lYW4oKSIsICJjaGVjayI6IHt9LCAieHAiOiA1MH0sICJtbDE0IjogeyJzb2x1dGlvbiI6ICJmcm9tIHNrbGVhcm4ubW9kZWxfc2VsZWN0aW9uIGltcG9ydCB0cmFpbl90ZXN0X3NwbGl0XG5mcm9tIHNrbGVhcm4uZW5zZW1ibGUgaW1wb3J0IFJhbmRvbUZvcmVzdENsYXNzaWZpZXJcbmZyb20gc2tsZWFybi5tZXRyaWNzIGltcG9ydCBhY2N1cmFjeV9zY29yZVxuXG5kYXRvcyA9IHBkLmdldF9kdW1taWVzKGNodXJuLCBjb2x1bW5zPVsnY29udHJhdG8nXSwgZHJvcF9maXJzdD1UcnVlLCBkdHlwZT1pbnQpXG5YID0gZGF0b3MuZHJvcChjb2x1bW5zPVsnYWJhbmRvbmEnXSlcbnkgPSBkYXRvc1snYWJhbmRvbmEnXVxuWF90cmFpbiwgWF90ZXN0LCB5X3RyYWluLCB5X3Rlc3QgPSB0cmFpbl90ZXN0X3NwbGl0KFgsIHksIHRlc3Rfc2l6ZT0wLjI1LCByYW5kb21fc3RhdGU9NDIsIHN0cmF0aWZ5PXkpXG5ib3NxdWUgPSBSYW5kb21Gb3Jlc3RDbGFzc2lmaWVyKG5fZXN0aW1hdG9ycz0xMDAsIHJhbmRvbV9zdGF0ZT00MilcbmJvc3F1ZS5maXQoWF90cmFpbiwgeV90cmFpbilcbnByaW50KHBkLlNlcmllcyhib3NxdWUuZmVhdHVyZV9pbXBvcnRhbmNlc18sIGluZGV4PVguY29sdW1ucykucm91bmQoMykuc29ydF92YWx1ZXMoYXNjZW5kaW5nPUZhbHNlKSlcbnJlc3VsdGFkbyA9IGFjY3VyYWN5X3Njb3JlKHlfdGVzdCwgYm9zcXVlLnByZWRpY3QoWF90ZXN0KSkiLCAiY2hlY2siOiB7fSwgInhwIjogNTB9fQ=='
_TABLAS = ['casas', 'churn']

import base64 as _b64, json as _json, re as _re, warnings as _w
_w.simplefilter("ignore")
_SOL = _json.loads(_b64.b64decode(_SOL_B64).decode("utf-8"))

def reiniciar_datos():
    """Vuelve a cargar todas las tablas limpias (cada ejercicio empieza de cero)."""
    globals().pop("resultado", None)
    exec(_SETUP, globals())
    pq_prepare()

def _codigo_celda():
    try:
        src = get_ipython().history_manager.input_hist_raw[-1]
    except Exception:
        return ""
    src = "\n".join(l for l in src.splitlines() if "comprobar(" not in l)
    return "\n".join(_re.sub(r"#.*$", "", l) for l in src.splitlines())

def comprobar(ej):
    info = _SOL[ej]
    def esperado():
        exp_ns = {}
        exec(_SETUP, exp_ns)
        exec(info["solution"], exp_ns)
        return exp_ns
    g = globals()
    g["_pq_code"] = _codigo_celda()
    ok, msg = pq_check(g, esperado, info["check"])
    if ok:
        from IPython.display import HTML, display
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#0f9d6e;color:#fff;'
                     'font:600 15px system-ui">🎉 ¡Correcto! +' + str(info["xp"]) + ' XP · Sigue así 🐼</div>'))
    else:
        from IPython.display import HTML, display
        import html as _h
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#b4235a;color:#fff;'
                     'font:500 15px system-ui">❌ Todavía no. ' + _h.escape(msg) + '</div>'))
    if "resultado" in g:
        return g["resultado"]

def ver_solucion(ej):
    print(_SOL[ej]["solution"])

reiniciar_datos()
print("✅ Listo." + (" Datos cargados: " + ", ".join(_TABLAS) if _TABLAS else " Ya puedes empezar."))


---
# 📌 Preparar los datos

## 1. ¿Qué es Machine Learning?
`Básico` · **+10 XP**

**Machine Learning** (aprendizaje automático) consiste en que un programa **aprenda patrones a partir de ejemplos**, en lugar de escribir las reglas a mano.

En el **aprendizaje supervisado**, que es el más común, le das al modelo ejemplos con la respuesta correcta:
- **X (features o variables)**: los datos de entrada (metros cuadrados, habitaciones, antigüedad…);
- **y (target u objetivo)**: lo que quieres predecir (el precio).

El modelo aprende la relación entre X e y, y después **predice y para casos nuevos**.

Hay dos grandes tipos de problema:
- **Regresión**: predecir un **número** (precio, ventas, temperatura).
- **Clasificación**: predecir una **categoría** (¿abandona o no?, ¿spam o no?).

En este módulo usarás **scikit-learn** (`sklearn`), la librería estándar de ML en Python, y dos datasets: `casas` (precios de viviendas) y `churn` (clientes que abandonan un servicio).

```python
X = df[["col1", "col2"]]    # DataFrame (doble corchete)
y = df["objetivo"]          # Series
```

> ⚠️ Por convención, `X` va en **mayúscula** (es una matriz) e `y` en **minúscula** (es un vector).

### 🎯 Tu misión
Prepara los datos del problema de precios: guarda en `X` las columnas `m2`, `habitaciones` y `antiguedad` de `casas`, y en `y` la columna `precio`.

<details><summary>💡 Pista</summary>

`X = casas[['m2', 'habitaciones', 'antiguedad']]; y = casas['precio']`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

X = 
y = 

comprobar("ml01")

## 2. Entrenamiento y prueba
`Intermedio` · **+20 XP**

Si evalúas un modelo con los **mismos datos con los que aprendió**, es como darle a un estudiante el examen con las respuestas: sacará un 10 sin haber aprendido nada. Por eso los datos se dividen en dos partes:

- **train (entrenamiento)**: el modelo aprende con esto (normalmente el 70-80%);
- **test (prueba)**: se reserva para medir cómo funciona con datos que **nunca vio**.

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
```

- `test_size=0.2`: un 20% para test;
- `random_state=42`: fija la semilla para que la división sea **siempre la misma** (reproducible).

> ⚠️ Respeta el orden de las 4 variables de salida: `X_train, X_test, y_train, y_test`.

### 🎯 Tu misión
Divide `X` e `y` con `train_test_split`: **20% para test** y `random_state=42`. Guarda `X_test` en `resultado`.

<details><summary>💡 Pista</summary>

`X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

from sklearn.model_selection import train_test_split

X = casas[['m2', 'habitaciones', 'antiguedad']]
y = casas['precio']

# divide aquí

resultado = X_test

comprobar("ml02")

## 3. Variables categóricas
`Avanzado` · **+35 XP**

Los modelos solo entienden **números**. Una columna de texto como `barrio` (`Centro`, `Norte`, `Sur`) hay que convertirla. La técnica más común es el **one-hot encoding**: se crea una columna de 0/1 por cada categoría.

```python
pd.get_dummies(df, columns=["barrio"], drop_first=True, dtype=int)
```

| barrio | → | barrio_Norte | barrio_Sur |
|---|---|---|---|
| Centro | | 0 | 0 |
| Norte | | 1 | 0 |
| Sur | | 0 | 1 |

`drop_first=True` elimina la primera categoría (`Centro`), porque es redundante: si Norte y Sur valen 0, tiene que ser Centro.

> ⚠️ **Error común:** convertir las categorías a 1, 2, 3. El modelo creería que "Sur" vale el triple que "Centro", un orden que no existe.

### 🎯 Tu misión
Aplica one-hot encoding a la columna `barrio` de `casas` con `pd.get_dummies`, usando `drop_first=True` y `dtype=int`. Guarda el DataFrame completo.

<details><summary>💡 Pista</summary>

`pd.get_dummies(casas, columns=['barrio'], drop_first=True, dtype=int)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

resultado = 

comprobar("ml03")

---
# 📌 Regresión: predecir números

## 4. Entrena tu primer modelo
`Intermedio` · **+20 XP**

Todos los modelos de scikit-learn siguen **el mismo patrón de 3 pasos**:

```python
from sklearn.linear_model import LinearRegression

modelo = LinearRegression()      # 1. crear el modelo
modelo.fit(X_train, y_train)     # 2. entrenar: aprende de los ejemplos
modelo.predict(X_test)           # 3. predecir casos nuevos
```

La **regresión lineal** busca la fórmula:

```
precio = intercepto + coef1·m2 + coef2·habitaciones + coef3·antigüedad
```

Después de entrenar, `modelo.coef_` tiene los coeficientes: **cuánto cambia el precio por cada unidad** de cada variable. Por ejemplo, un coeficiente de 1800 en `m2` significa que cada metro cuadrado suma unos 1800 al precio.

> 💡 Una vez aprendas este patrón, cambiar de modelo (árboles, bosques, redes…) es cambiar **una sola línea**.

### 🎯 Tu misión
Crea una `LinearRegression`, entrénala con `X_train` e `y_train` y guarda sus coeficientes (`modelo.coef_`) en `resultado`.

<details><summary>💡 Pista</summary>

`modelo = LinearRegression(); modelo.fit(X_train, y_train)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

from sklearn.model_selection import train_test_split

X = casas[['m2', 'habitaciones', 'antiguedad']]
y = casas['precio']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.linear_model import LinearRegression

modelo = 
# entrena aquí

resultado = modelo.coef_

comprobar("ml04")

## 5. Hacer predicciones
`Intermedio` · **+20 XP**

Con el modelo ya entrenado, `modelo.predict(X_nuevo)` devuelve un **array con una predicción por fila**.

```python
predicciones = modelo.predict(X_test)
predicciones[:3]      # las 3 primeras
```

Para evaluar, se comparan las predicciones con los valores reales (`y_test`), que el modelo **nunca vio** durante el entrenamiento.

> 💡 La entrada de `predict` debe tener **las mismas columnas y en el mismo orden** que usaste en `fit`.

### 🎯 Tu misión
Usa el modelo entrenado para predecir los precios de `X_test`. Guarda el array de predicciones en `resultado`.

<details><summary>💡 Pista</summary>

`modelo.predict(X_test)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

from sklearn.model_selection import train_test_split

X = casas[['m2', 'habitaciones', 'antiguedad']]
y = casas['precio']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.linear_model import LinearRegression
modelo = LinearRegression()
modelo.fit(X_train, y_train)

resultado = 

comprobar("ml05")

## 6. ¿Qué tan bien predice? MAE
`Intermedio` · **+20 XP**

¿Cuánto se equivoca el modelo? En regresión, la métrica más intuitiva es el **MAE** (*Mean Absolute Error*, error absoluto medio): el **promedio de lo que se equivoca**, en las mismas unidades que el objetivo.

```python
from sklearn.metrics import mean_absolute_error
mean_absolute_error(y_test, predicciones)    # por ejemplo 12500 -> se equivoca ~12.500 de media
```

Otras métricas habituales:
- **RMSE**: castiga más los errores grandes;
- **R²** (lo verás a continuación): qué proporción de la variación explica el modelo.

> 💼 **En el trabajo:** "el modelo se equivoca en promedio 12.500 € por casa" es una frase que cualquier persona del negocio entiende.

### 🎯 Tu misión
Calcula el **MAE** de las predicciones sobre el conjunto de test y guárdalo en `resultado`.

<details><summary>💡 Pista</summary>

`mean_absolute_error(y_test, predicciones): primero los valores reales`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

from sklearn.model_selection import train_test_split

X = casas[['m2', 'habitaciones', 'antiguedad']]
y = casas['precio']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.linear_model import LinearRegression
modelo = LinearRegression()
modelo.fit(X_train, y_train)
predicciones = modelo.predict(X_test)

from sklearn.metrics import mean_absolute_error
resultado = 

comprobar("ml06")

## 7. Coeficiente R²
`Avanzado` · **+35 XP**

El **R²** (coeficiente de determinación) indica qué proporción de la variación del objetivo **explica** el modelo:

- **1.0**: predicción perfecta;
- **0.0**: el modelo no es mejor que predecir siempre la media;
- **negativo**: peor que la media (¡algo va mal!).

```python
from sklearn.metrics import r2_score
r2_score(y_test, predicciones)
```

Muchos modelos también lo calculan directamente con `modelo.score(X_test, y_test)`.

> 💡 Un R² "bueno" depende del problema: 0.9 es excelente para precios de casas, y en fenómenos humanos (ventas, comportamiento) un 0.5 ya puede ser útil.

### 🎯 Tu misión
Calcula el **R²** del modelo sobre el conjunto de test con `r2_score` y guárdalo en `resultado`.

<details><summary>💡 Pista</summary>

`r2_score(y_test, predicciones)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

from sklearn.model_selection import train_test_split

X = casas[['m2', 'habitaciones', 'antiguedad']]
y = casas['precio']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.linear_model import LinearRegression
modelo = LinearRegression()
modelo.fit(X_train, y_train)
predicciones = modelo.predict(X_test)

from sklearn.metrics import r2_score
resultado = 

comprobar("ml07")

## 8. Predecir un caso nuevo
`Intermedio` · **+20 XP**

El objetivo final de un modelo es **predecir casos reales nuevos**. Se prepara un DataFrame con **las mismas columnas** que usaste para entrenar:

```python
nueva = pd.DataFrame([{"m2": 120, "habitaciones": 3, "antiguedad": 5}])
modelo.predict(nueva)       # array([287000.])
modelo.predict(nueva)[0]    # el número suelto
```

`predict` siempre devuelve un **array**, aunque sea para una sola fila; con `[0]` obtienes el valor.

> 💼 **En el trabajo:** así funcionan las APIs de predicción: llega un caso nuevo, se arma su fila de datos y el modelo devuelve la respuesta.

### 🎯 Tu misión
Predice el precio de una casa de **90 m²**, **3 habitaciones** y **10 años** de antigüedad. Guarda el **número** (no el array) en `resultado`.

<details><summary>💡 Pista</summary>

`modelo.predict(nueva)[0]`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

from sklearn.model_selection import train_test_split

X = casas[['m2', 'habitaciones', 'antiguedad']]
y = casas['precio']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.linear_model import LinearRegression
modelo = LinearRegression()
modelo.fit(X_train, y_train)
predicciones = modelo.predict(X_test)

nueva = pd.DataFrame([{'m2': 90, 'habitaciones': 3, 'antiguedad': 10}])
resultado = 

comprobar("ml08")

---
# 📌 Clasificación: predecir categorías

## 9. Regresión logística
`Avanzado` · **+35 XP**

Ahora un problema de **clasificación**: predecir si un cliente **abandona** el servicio (`abandona` = 1) o no (0). Es el famoso problema de **churn**, uno de los casos de ML más comunes en las empresas.

A pesar de su nombre, la **regresión logística** es un modelo de **clasificación**: calcula la **probabilidad** de que ocurra algo y decide 1 si pasa del 50%.

```python
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train, y_train)
pred = modelo.predict(X_test)
accuracy_score(y_test, pred)      # proporción de aciertos (0 a 1)
```

- `max_iter=1000` le da al algoritmo más iteraciones para converger.
- `stratify=y` en `train_test_split` mantiene la misma proporción de 1 y 0 en train y en test.

### 🎯 Tu misión
Entrena una `LogisticRegression(max_iter=1000)`, predice sobre `X_test` y guarda la **accuracy** (proporción de aciertos) en `resultado`.

<details><summary>💡 Pista</summary>

`modelo = LogisticRegression(max_iter=1000); modelo.fit(...); pred = modelo.predict(X_test); accuracy_score(y_test, pred)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

from sklearn.model_selection import train_test_split

X = churn[['meses_cliente', 'cargo_mensual', 'reclamos']]
y = churn['abandona']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

resultado = 

comprobar("ml09")

## 10. Matriz de confusión
`Avanzado` · **+35 XP**

La accuracy puede **engañar**: si solo el 10% de los clientes abandona, un modelo que siempre dice "no abandona" acierta el 90% sin servir para nada. La **matriz de confusión** muestra **dónde** se equivoca:

|  | predice 0 | predice 1 |
|---|---|---|
| **real 0** | verdaderos negativos | falsos positivos |
| **real 1** | falsos negativos | verdaderos positivos |

```python
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, pred)
# [[40  5]
#  [12 18]]
```

Métricas que salen de aquí:
- **precision**: de los que predije como 1, ¿cuántos lo eran?
- **recall**: de los que eran 1, ¿cuántos detecté?

> 💼 **En el trabajo:** en churn importa mucho el **recall**. Cada cliente que se va sin que lo detectes es dinero perdido.

### 🎯 Tu misión
Calcula la **matriz de confusión** de las predicciones del modelo logístico y guárdala en `resultado`.

<details><summary>💡 Pista</summary>

`confusion_matrix(y_test, pred)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

from sklearn.model_selection import train_test_split

X = churn[['meses_cliente', 'cargo_mensual', 'reclamos']]
y = churn['abandona']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train, y_train)
pred = modelo.predict(X_test)

from sklearn.metrics import confusion_matrix
resultado = 

comprobar("ml10")

## 11. Árbol de decisión
`Avanzado` · **+35 XP**

Un **árbol de decisión** aprende una serie de preguntas del tipo *sí/no*:

```
¿reclamos > 2?
 ├─ sí → ¿meses_cliente < 12? → abandona
 └─ no → no abandona
```

Es muy **interpretable**: puedes explicarle a cualquiera por qué predijo lo que predijo.

```python
from sklearn.tree import DecisionTreeClassifier
arbol = DecisionTreeClassifier(max_depth=3, random_state=0)
```

`max_depth` limita cuántas preguntas seguidas puede hacer. Sin límite, el árbol **memoriza** los datos de entrenamiento (**overfitting**, sobreajuste) y luego falla con datos nuevos.

### 🎯 Tu misión
Entrena un `DecisionTreeClassifier(max_depth=3, random_state=0)` y guarda su **accuracy** sobre el conjunto de test en `resultado`.

<details><summary>💡 Pista</summary>

`arbol = DecisionTreeClassifier(max_depth=3, random_state=0); arbol.fit(...); accuracy_score(y_test, arbol.predict(X_test))`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

from sklearn.model_selection import train_test_split

X = churn[['meses_cliente', 'cargo_mensual', 'reclamos']]
y = churn['abandona']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

resultado = 

comprobar("ml11")

---
# 📌 Buenas prácticas

## 12. Escalado y Pipeline
`Pro` · **+50 XP**

Algunos modelos, como **KNN** (vecinos más cercanos), miden **distancias** entre puntos. Si una variable va de 0 a 120 (cargo) y otra de 0 a 5 (reclamos), la primera dominaría. Por eso hay que **escalar**: `StandardScaler` estandariza cada columna (media 0 y desviación 1), como hiciste a mano en NumPy.

Un **Pipeline** encadena el preprocesado y el modelo en **un solo objeto**:

```python
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
pipe.fit(X_train, y_train)      # escala Y entrena
pipe.predict(X_test)            # escala con lo aprendido en train Y predice
```

> ⚠️ **Error grave muy común:** escalar **todos** los datos antes de dividir. Así se filtra información del test al entrenamiento (*data leakage*). El Pipeline lo evita automáticamente.

### 🎯 Tu misión
Crea un pipeline con `StandardScaler()` y `KNeighborsClassifier(n_neighbors=5)`, entrénalo y guarda su **accuracy** en test en `resultado`.

<details><summary>💡 Pista</summary>

`pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)); pipe.fit(...)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

from sklearn.model_selection import train_test_split

X = churn[['meses_cliente', 'cargo_mensual', 'reclamos']]
y = churn['abandona']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

pipe = 
resultado = 

comprobar("ml12")

## 13. Validación cruzada
`Pro` · **+50 XP**

Una sola división train/test puede salir **"con suerte"** (o sin ella). La **validación cruzada** (*cross-validation*) divide los datos en *k* partes (*folds*) y entrena *k* veces, usando cada vez una parte distinta como test:

```
Fold 1: [TEST][train][train][train][train]
Fold 2: [train][TEST][train][train][train]
...
```

```python
from sklearn.model_selection import cross_val_score
scores = cross_val_score(modelo, X, y, cv=5)   # 5 accuracies
scores.mean()                                  # la estimación final
```

Obtienes una medida **más confiable** del rendimiento y además ves cuánto varía (`scores.std()`).

> 💼 **En el trabajo:** es la forma estándar de comparar modelos antes de elegir uno.

### 🎯 Tu misión
Evalúa un `DecisionTreeClassifier(max_depth=3, random_state=0)` con **validación cruzada de 5 folds** sobre `X` e `y` completos, y guarda la **media** de los scores.

<details><summary>💡 Pista</summary>

`cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=0), X, y, cv=5).mean()`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier

X = churn[['meses_cliente', 'cargo_mensual', 'reclamos']]
y = churn['abandona']

resultado = 

comprobar("ml13")

## 14. Boss final: proyecto de churn
`Pro` · **+50 XP**

¡Proyecto completo, como en un trabajo real! El flujo de un proyecto de ML es:

1. **Preparar**: elegir las variables y codificar las categóricas.
2. **Dividir**: train/test.
3. **Entrenar**: aquí usarás un **Random Forest**, un "bosque" de muchos árboles que votan; suele ser muy preciso sin mucho ajuste.
4. **Evaluar** con datos no vistos.

```python
from sklearn.ensemble import RandomForestClassifier
bosque = RandomForestClassifier(n_estimators=100, random_state=42)
```

Después de entrenar, `bosque.feature_importances_` indica **qué variables pesaron más**. ¡Imprímelo con `print` para verlo!

> 💼 Con esto ya puedes armar tu primer proyecto de portfolio: "Modelo de predicción de abandono de clientes".

### 🎯 Tu misión
1. Aplica `pd.get_dummies` a `churn` con `columns=['contrato']`, `drop_first=True` y `dtype=int` → `datos`.
2. `X` = todas las columnas de `datos` **menos** `abandona`; `y` = `datos['abandona']`.
3. Divide con `test_size=0.25`, `random_state=42` y `stratify=y`.
4. Entrena `RandomForestClassifier(n_estimators=100, random_state=42)`.
5. Guarda la **accuracy** en test en `resultado`.

<details><summary>💡 Pista</summary>

`datos = pd.get_dummies(...); X = datos.drop(columns=['abandona']); ...; accuracy_score(y_test, bosque.predict(X_test))`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Preparar

# 2. Dividir

# 3. Entrenar

# 4. Evaluar
resultado = 

comprobar("ml14")

---
# 🏁 ¡Módulo completado!
Vuelve a [PandasQuest](https://rosalesluciano.github.io/pandas-quest/) para sumar tus XP y seguir con el siguiente módulo. 🚀